# Fractional delayed fault model — run the whole study from GitHub (Colab, parallel)

Run the cells in order. The repository is cloned, **all tasks run in one process pool on every CPU core**, and everything is saved in your Google Drive folder `MyDrive/fault_study/` (resumable: if Colab disconnects, run the same cells again and finished tasks are skipped).

Tip: Runtime ▸ Change runtime type ▸ more CPU cores (if your plan offers a high-RAM/many-core runtime) makes the study proportionally faster; the GPU is not used.

In [ ]:
# 1. Google Drive (all results are written here)
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/fault_study'
for d in ('out', 'figs', 'tabs', 'data'):
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
print('results folder:', DRIVE)

In [ ]:
# 2. get the repository (public: no token needed; private: you will be asked for a GitHub token with read access)
import subprocess, getpass, os
REPO = 'yamenetoo/fractional-delayed-fault-model'
if os.path.isdir('/content/repo/.git'):
    print(subprocess.run('cd /content/repo && git pull -q', shell=True, capture_output=True, text=True).stdout or 'repo updated')
else:
    r = subprocess.run(f'git clone -q https://github.com/{REPO} /content/repo', shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        tok = getpass.getpass('Repository is private - paste a GitHub token (not shown): ')
        r = subprocess.run(f'git clone -q https://{tok}@github.com/{REPO} /content/repo', shell=True, capture_output=True, text=True)
        print(r.stderr.replace(tok, '***') if r.returncode else 'cloned')
    else:
        print('cloned')
print(subprocess.run('cd /content/repo && git log --oneline | head -3', shell=True, capture_output=True, text=True).stdout)

In [ ]:
# 3. dependencies + link the result folders of the repo to Drive
!pip -q install numba openpyxl
!apt-get -qq install -y libgeos-dev >/dev/null
!pip -q install cartopy
import os, multiprocessing
%cd /content/repo/stats
for d in ('out', 'figs', 'tabs', 'data'):
    !rm -rf {d}
    !ln -sfn {DRIVE}/{d} {d}
NCPU = multiprocessing.cpu_count()
print('CPU cores:', NCPU)

In [ ]:
# 4. size of the study
MODE = 'FULL'   # 'QUICK' = few-minute check, 'FULL' = the study of the article
CFG = dict(QUICK=dict(R1=3, B1=6, R2=3, B2=6, M=2), FULL=dict(R1=24, B1=40, R2=16, B2=24, M=12))[MODE]
CFG

In [ ]:
# 5. RUN EVERYTHING IN PARALLEL (S1 and S2 replicates, bootstrap null, robustness) in one pool of NCPU processes
#    ~1.5 h on 2 cores for FULL; progress lines show finished tasks. Safe to re-run after a disconnect.
!python -u run_all.py --procs {NCPU} --R1 {CFG['R1']} --B1 {CFG['B1']} --R2 {CFG['R2']} --B2 {CFG['B2']} --M {CFG['M']} --npart 100 --out out

In [ ]:
# 6. real-fault map (cartopy + GEM database). Choose the two faults by name substring (see the printed list)
os.environ['FAULT1'] = 'EMME'
os.environ['FAULT2'] = 'SHARE'
!python fault_map.py

In [ ]:
# 7. basic figures, result figures/tables, Excel workbook
!python fig_basic.py
!python analyze.py
!python export_excel.py
!cp summary.json results.xlsx {DRIVE}/

In [ ]:
# 8. article bundle: unzip it anywhere and run pdflatex in paper/
import zipfile, glob
with zipfile.ZipFile(f'{DRIVE}/article_bundle.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob('/content/repo/paper/*.tex'):
        z.write(f, 'paper/' + os.path.basename(f))
    z.write('/content/repo/stats/schematic.tex', 'stats/schematic.tex')
    for sub in ('figs', 'tabs'):
        for f in glob.glob(f'{DRIVE}/{sub}/*'):
            z.write(f, f'stats/{sub}/' + os.path.basename(f))
print('saved', f'{DRIVE}/article_bundle.zip')
print(open(f'{DRIVE}/summary.json').read()[:1500])

In [ ]:
# 9. (optional) compile the PDF here: installs LaTeX (~3-5 min)
!apt-get -qq install -y texlive-latex-recommended texlive-latex-extra texlive-fonts-recommended > /dev/null
%cd /content/repo/paper
!pdflatex -interaction=nonstopmode article.tex > /dev/null; pdflatex -interaction=nonstopmode article.tex | tail -3
!cp article.pdf {DRIVE}/article.pdf
%cd /content/repo/stats

In [ ]:
# 10. preview of the result figures
import glob, fitz
from IPython.display import Image, display
get_ipython().system('pip -q install pymupdf')
import fitz
for f in sorted(glob.glob('figs/*.pdf')):
    png = f[:-4] + '.png'
    fitz.open(f)[0].get_pixmap(dpi=100).save(png)
    print(f); display(Image(png))